# 📄 Energy & Gas Bill Identifier & Renamer

This Jupyter notebook identifies PDF energy bills located in `landing/energy_australia/gas` and `landing/globird` and renames them according to a standardized convention.

---

### 🏷️ Naming Convention
```
<energycompany>_<utilitytype>_<issuedate>.pdf
```

- **`energycompany`**: `energyaustralia` or `globird`
- **`utilitytype`**: `elec`, `gas`, or `both`
- **`issuedate`**: `YYYYMMDD` (e.g. `20240415`)

---
### 🚦 Approval Workflow
1. Run **Cells 1 to 3** to extract bill metadata and preview proposed filenames.
2. Review the preview table in **Step 3**.
3. In **Step 4**, set `APPROVE_AND_RENAME = True` and run the final cell to execute renaming.


In [ ]:
# Step 1: Import Required Libraries & Configuration
import os
import re
from pathlib import Path
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv
import sys

# Add project root directory to sys.path
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
# Now imports from  work seamlessly!
from src.config import BASE_DIR, LANDING_DIR
from src.extractors import parse_bill

# Load environment variables from .env file
load_dotenv(BASE_DIR / '.env')

# Define target directories
LANDING_DIR = LANDING_DIR
TARGET_DIRS = [
    LANDING_DIR / "energy_australia",
    LANDING_DIR / "globird"
]

print(f"LANDING_DIR: {LANDING_DIR}")
print("✓ Environment, target directories, and UtilityBill extractor configured.")


In [ ]:
# Step 2: Define PDF Parsing & Metadata Extraction Logic Using UtilityBill Classes

def parse_date(date_str):
    """Parse raw date string into datetime object."""
    if not date_str:
        return None
    date_str = date_str.strip()
    for fmt in ('%d-%b-%Y', '%d %b %Y', '%d%b%Y', '%d %B %Y', '%d/%m/%Y'):
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            pass
    return None

def extract_bill_info(filepath):
    """
    Extract company, utility type, and issue date from PDF bill using src.extractors (UtilityBill classes).
    """
    records = parse_bill(filepath)
    if not records:
        return "unknown", "unknown", None

    rec = records[0]
    company = rec.get("Provider", "unknown").lower()
    
    if company == "globird":
        utility = "both"
    else:
        utility = "elec" if rec.get("Utility") == "Electricity" else "gas"

    issue_dt = parse_date(rec.get("Issue Date"))
    return company, utility, issue_dt

print("✓ Metadata extraction function ready using UtilityBill classes.")


## Step 3: Scan Bills & Preview Proposed Names

Run the cell below to parse all PDF bills and inspect the proposed filenames in the DataFrame preview.

In [ ]:
records = []

for dir_path in TARGET_DIRS:
    if not dir_path.exists():
        print(f"Warning: Directory {dir_path} does not exist.")
        continue
    # Match both .pdf and .PDF files
    pdf_files = [f for f in sorted(dir_path.glob("*")) if f.suffix.lower() == ".pdf"]

    for pdf_file in pdf_files:
        company, utility, issue_dt = extract_bill_info(pdf_file)
        
        date_str = issue_dt.strftime("%Y%m%d") if issue_dt else "UNKNOWN_DATE"
        proposed_name = f"{company}_{utility}_{date_str}.pdf"
        
        records.append({
            "Directory": str(dir_path.relative_to(BASE_DIR)),
            "Current Filename": pdf_file.name,
            "Company": company,
            "Utility": utility,
            "Issue Date": issue_dt.strftime("%Y-%m-%d") if issue_dt else "Unknown",
            "Proposed Filename": proposed_name,
            "Full Current Path": pdf_file,
        })

df_preview = pd.DataFrame(records)
print(f"Total PDF files detected: {len(df_preview)}\n")

# Display preview dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
df_preview[["Directory", "Current Filename", "Company", "Utility", "Issue Date", "Proposed Filename"]]


## Step 4: Approve & Execute File Renaming

Review the table above. If you approve the proposed filenames, set `APPROVE_AND_RENAME = True` below and execute this cell to rename the files on disk.

In [ ]:
# --- APPROVAL FLAG ---
# Set to True to execute file renaming, or False to perform a dry run.
APPROVE_AND_RENAME = True


def execute_renaming(df, execute=False):
    renamed_count = 0
    skipped_count = 0
    dir_target_counts = {}

    print("=" * 70)
    if execute:
        print("🚀 EXECUTING FILE RENAMING...")
    else:
        print("🔍 DRY RUN MODE (No files will be changed)...")
    print("=" * 70)

    for idx, row in df.iterrows():
        curr_path = row["Full Current Path"]
        parent_dir = curr_path.parent
        
        company = row["Company"]
        utility = row["Utility"]
        issue_date_str = row["Issue Date"].replace("-", "") if row["Issue Date"] != "Unknown" else "UNKNOWN"
        
        base_target = f"{company}_{utility}_{issue_date_str}"
        ext = ".pdf"
        
        # Prevent collisions for identical target names in the same folder
        key = (parent_dir, base_target)
        count = dir_target_counts.get(key, 0)
        dir_target_counts[key] = count + 1
        
        target_name = f"{base_target}{ext}" if count == 0 else f"{base_target}_{count}{ext}"
        target_path = parent_dir / target_name
        
        if curr_path.name == target_name:
            print(f"  [SKIP] Already correctly named: {curr_path.name}")
            skipped_count += 1
            continue
            
        if execute:
            curr_path.rename(target_path)
            print(f"  ✓ RENAMED: {curr_path.name}  ->  {target_name}")
            renamed_count += 1
        else:
            print(f"  [WOULD RENAME] {curr_path.name}  ->  {target_name}")
            renamed_count += 1

    print("=" * 70)
    if execute:
        print(f"✅ SUCCESS: {renamed_count} files renamed ({skipped_count} skipped).")
    else:
        print(f"ℹ️ DRY RUN COMPLETE: {renamed_count} files ready to be renamed. Set APPROVE_AND_RENAME = True to apply.")
    print("=" * 70)


# Run renaming execution
execute_renaming(df_preview, execute=APPROVE_AND_RENAME)
